# Fusion des données pour la création de data set

In [8]:
import pandas as pd

## Gestion des fichiers météos

In [31]:
meteo = pd.read_csv("./climat_45/climat_mensuel_45_total.csv", sep=";")

meteo = meteo[["AAAAMM", "RR", "TMM", "ETP","LAT","LON"]]
meteo = meteo.rename(columns={
    "AAAAMM": "time",
    "RR": "PRELIQ_Q",
    "TMM": "T_Q",
    "LAT": "lat",
    "LON": "lon"
})

meteo["time"] = pd.to_datetime(meteo["time"].astype(str), format="%Y%m")

meteo_month = meteo.sort_values("time").reset_index(drop=True)
print(meteo_month.head())

        time  PRELIQ_Q  T_Q  ETP        lat       lon
0 1871-12-01       NaN  NaN  NaN  48.038333  3.060000
1 1871-12-01       NaN  NaN  NaN  47.901167  1.928167
2 1871-12-01       NaN  NaN  NaN  47.903333  1.906667
3 1871-12-01       NaN  NaN  NaN  47.885000  2.445000
4 1871-12-01       NaN  NaN  NaN  47.856667  2.573333


## Gestion des fichiers ades

In [17]:
nappe = pd.read_csv("./nappe_piezometres/nappe_02927X1013_P.csv", sep=";")

# Renommer la date
nappe = nappe.rename(columns={
    "date_mesure": "time"
})

# Conversion en datetime
nappe["time"] = pd.to_datetime(nappe["time"])

# Passage en mois
nappe["month"] = nappe["time"].dt.to_period("M")

# Agrégation mensuelle par piézomètre
nappe_month = nappe.groupby(["code_bss", "month"]).agg({
    "niveau_nappe_eau": "mean",  # niveau moyen mensuel
    "lon": "first",              # coordonnée fixe
    "lat": "first"
}).reset_index()

# Revenir à une date classique (premier jour du mois)
nappe_month["time"] = nappe_month["month"].dt.to_timestamp()

# Nettoyage
nappe_month = nappe_month.drop(columns="month")

print(nappe_month.head())

       code_bss  niveau_nappe_eau        lon       lat       time
0  02927X1013/P           111.190  48.282515  2.018695 1972-07-01
1  02927X1013/P           111.175  48.282515  2.018695 1972-08-01
2  02927X1013/P           111.158  48.282515  2.018695 1972-09-01
3  02927X1013/P           111.140  48.282515  2.018695 1972-10-01
4  02927X1013/P           111.125  48.282515  2.018695 1972-11-01


## Gestion des fichiers ETP

In [29]:
import pandas as pd

etp = pd.read_csv("./etp_fao_hargreaves/etp_france_total.csv", sep=";")

# Conversion date (format 19700101)
etp["time"] = pd.to_datetime(etp["DATE"].astype(str), format="%Y%m%d")

etp = etp.rename(columns={
    "lat_dg": "lat",
    "lon_dg": "lon"
})

# Passage en mois
etp["month"] = etp["time"].dt.to_period("M")

# Agrégation mensuelle par maille (lat/lon)
etp_month = etp.groupby(["lat", "lon", "month"]).agg({
    "ETP_Q_H0175": "sum"   # ETP mensuelle = somme des jours
}).reset_index()

# Revenir en datetime
etp_month["time"] = etp_month["month"].dt.to_timestamp()

# Nettoyage
etp_month = etp_month.drop(columns=["month"])

print(etp_month.head())


      lat      lon  ETP_Q_H0175       time
0  47.454  1.54152       10.788 1970-01-01
1  47.454  1.54152       18.924 1970-02-01
2  47.454  1.54152       34.758 1970-03-01
3  47.454  1.54152       55.556 1970-04-01
4  47.454  1.54152      100.874 1970-05-01


## Fusion

In [33]:
import pandas as pd
from scipy.spatial import cKDTree
import numpy as np

def nearest_point(df_points, df_target):
    """
    Pour chaque ligne de df_target, trouver la ligne la plus proche dans df_points
    et retourner les indices correspondants.
    """
    tree = cKDTree(df_points[["lat", "lon"]].values)
    dist, idx = tree.query(df_target[["lat", "lon"]].values)
    return idx

# Copie de nappe
df = nappe_month.copy()

# Dates présentes dans nappe
dates = df["time"].unique()

# Initialiser les colonnes ETP et météo
df["ETP_Q_H0175"] = np.nan
df["PRELIQ_Q"] = np.nan
df["T_Q"] = np.nan

# Pour chaque date, chercher les points les plus proches
for d in dates:
    mask = df["time"] == d
    
    # Sous-ensembles pour cette date
    df_day = df[mask]
    etp_day = etp_month[etp_month["time"] == d]
    meteo_day = meteo_month[meteo_month["time"] == d]
    
    # Fusion ETP
    if len(etp_day) > 0:
        idx_etp = nearest_point(etp_day, df_day)
        df.loc[mask, "ETP_Q_H0175"] = etp_day.iloc[idx_etp]["ETP_Q_H0175"].values
    
    # Fusion Météo
    if len(meteo_day) > 0:
        idx_met = nearest_point(meteo_day, df_day)
        df.loc[mask, "PRELIQ_Q"] = meteo_day.iloc[idx_met]["PRELIQ_Q"].values
        df.loc[mask, "T_Q"] = meteo_day.iloc[idx_met]["T_Q"].values

# Export final en CSV
output_file = "./nappe_complete.csv"
df.to_csv(output_file, sep=";", index=False)

print(f"Fichier final créé : {output_file}")


Fichier final créé : ./nappe_complete.csv


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Liste des fichiers CSV dans le dossier
dossier = "../data"
fichiers = [f for f in os.listdir(dossier) if f.endswith(".csv")]

# Menu déroulant pour choisir un fichier
dropdown = widgets.Dropdown(
    options=fichiers,
    description='Fichier:',
)

# Bouton pour générer le graphique
button = widgets.Button(description="Afficher graphique")

# Fonction pour afficher le graphique
def afficher_graphique(b):
    fichier_choisi = dropdown.value
    df = pd.read_csv(os.path.join(dossier, fichier_choisi))
    df.plot()  # ou df['colonne'].plot()
    plt.show()

button.on_click(afficher_graphique)

# Afficher les widgets
display(dropdown, button)


FileNotFoundError: [WinError 3] Le chemin d’accès spécifié est introuvable: './data'